# DON Exploration — Multi-Agent Exploration on the Four Rooms GridWorld

## Motivation

In `Exploration_test.ipynb` we validated a **multi-agent latent-space exploration**
strategy on a simple 10×10 dummy grid.  This notebook adapts the *exact same*
algorithm to the **real `FourRoomsGridWorld` environment** so that the collected
transitions can later be used to train the Deep Option Network (DON).

### Strategy recap

| Component | Role |
|-----------|------|
| **LatentSpaceManager** | Tracks running mean/std of encoded states; provides Latin-Hypercube-Sampling (LHS) initial positions spread across the state space. |
| **LatentExplorationPlanner** | At every step proposes a direction for each agent by balancing *novelty* (distance to archive), *repulsion* (push agents apart), *momentum* (memory smoothing), and *centering*. |
| **Identity encoder** | For now `z = state` (2-D position). A learned encoder can be swapped in later. |
| **Multiple parallel agents** | Each agent has its own env instance. The planner coordinates them to maximise state-space coverage. |

### Key adaptations for Four Rooms

| Aspect | Dummy grid | Four Rooms |
|--------|-----------|-------------|
| Grid size | 10 × 10 open box | 23 × 23 with internal walls & doors |
| Walls | None (clip only) | Handled by env substep collision |
| LHS init | Positions always valid | Must snap to nearest free cell |
| `repel_radius` | 1.0 | 2.0 (larger grid) |
| `step_size` | 0.5 | 1.0 (one-cell scale) |
| Action interface | Direct coord update | Continuous action ∈ [-1, 1]² via `env.step()` |

## 1 — Imports and path setup

In [ ]:
import sys
from pathlib import Path

# Walk up to find the `src/` directory and add it to sys.path
_nb_dir = Path('.').resolve()
for _p in [_nb_dir] + list(_nb_dir.parents):
    if (_p / 'src').is_dir():
        _src = str(_p / 'src')
        if _src not in sys.path:
            sys.path.insert(0, _src)
        break

import numpy as np
import matplotlib.pyplot as plt
from environments.fourrooms import FourRoomsGridWorld

print('FourRoomsGridWorld imported successfully')
print(f'sys.path includes: {_src}')

## 2 — Utility functions

These are identical to the helpers in `Exploration_test.ipynb`.

In [ ]:
def unit_vector(v, eps=1e-8):
    norm = np.linalg.norm(v, axis=-1, keepdims=True)
    return v / np.clip(norm, eps, None)


def running_mean_std(z_samples, axis=0, eps=1e-8):
    mu = np.mean(z_samples, axis=axis)
    sigma = np.std(z_samples, axis=axis) + eps
    return mu, sigma


def latin_hypercube_sampling(num_points, dim, low=-2.0, high=2.0, rng=None):
    if rng is None:
        rng = np.random.RandomState()
    cut = np.linspace(0, 1, num_points + 1)
    u = rng.rand(num_points, dim)
    a = cut[:num_points]
    b = cut[1:num_points + 1]
    rdpoints = u * (b - a)[:, None] + a[:, None]
    H = np.zeros_like(rdpoints)
    for j in range(dim):
        order = rng.permutation(num_points)
        H[:, j] = rdpoints[order, j]
    return low + (high - low) * H

## 3 — LatentSpaceManager

Tracks running statistics of the latent (= state) distribution and provides
LHS-based initial positions.  Identical to `Exploration_test`.

In [ ]:
class LatentSpaceManager:
    def __init__(self, latent_dim, init_c=2.0, rng=None):
        self.latent_dim = latent_dim
        self.init_c = init_c
        self.rng = np.random.RandomState() if rng is None else rng
        self.mu = np.zeros(latent_dim)
        self.sigma = np.ones(latent_dim)

    def update_stats(self, z_batch):
        self.mu, self.sigma = running_mean_std(z_batch)

    def to_standardized(self, z):
        return (z - self.mu) / self.sigma

    def from_standardized(self, z_tilde):
        return self.mu + self.sigma * z_tilde

    def lhs_initial_latents(self, num_agents):
        z_tilde = latin_hypercube_sampling(
            num_points=num_agents,
            dim=self.latent_dim,
            low=-self.init_c,
            high=self.init_c,
            rng=self.rng
        )
        z = self.from_standardized(z_tilde)
        return z

## 4 — LatentExplorationPlanner

Identical to `Exploration_test`.  Each step, it proposes a direction for every
agent by scoring *K* candidate directions against novelty, spacing, and
centering objectives.

In [ ]:
class LatentExplorationPlanner:
    def __init__(
        self,
        latent_dim,
        num_candidates=16,
        center_weight=0.1,
        home_weight=0.1,
        memory_weight=0.5,
        repel_weight=0.2,
        novelty_weight=1.0,
        spacing_weight=1.0,
        center_align_weight=0.1,
        memory_smooth=0.5,
        repel_radius=1.0,
        step_size=0.5,
        rng=None,
    ):
        self.d = latent_dim
        self.K = num_candidates
        self.w_c = center_weight
        self.w_h = home_weight
        self.w_m = memory_weight
        self.w_r = repel_weight
        self.lambda_nov = novelty_weight
        self.lambda_space = spacing_weight
        self.lambda_center = center_align_weight
        self.gamma = memory_smooth
        self.repel_radius = repel_radius
        self.step_size = step_size
        self.rng = np.random.RandomState() if rng is None else rng

        self.z_home = None
        self.memory = None
        self.archive = []

    def reset_agents(self, z_init):
        self.z_home = np.array(z_init).copy()
        self.memory = np.zeros_like(z_init)
        self.archive = [z.copy() for z in z_init]

    def _latent_center(self, z_agents):
        return np.mean(z_agents, axis=0)

    def _repulsion_direction(self, z_agents, i):
        zi = z_agents[i]
        diff = zi - z_agents
        dist = np.linalg.norm(diff, axis=1, keepdims=True)
        dist[i] = np.inf
        mask = (dist < self.repel_radius).astype(np.float32)
        safe_dist = np.clip(dist, 1e-8, None)
        contrib = mask * (diff / safe_dist)
        vec = np.sum(contrib, axis=0)
        if np.allclose(vec, 0.0):
            return np.zeros_like(vec)
        return unit_vector(vec)

    def _novelty_score(self, z_candidate, k=5):
        if len(self.archive) == 0:
            return 0.0
        A = np.stack(self.archive, axis=0)
        diff = A - z_candidate[None, :]
        dist = np.linalg.norm(diff, axis=1)
        k = min(k, len(dist))
        idx = np.argpartition(dist, k - 1)[:k]
        return float(np.mean(dist[idx]))

    def _spacing_penalty(self, z_candidate, z_agents, i):
        diff = z_candidate[None, :] - z_agents
        dist = np.linalg.norm(diff, axis=1)
        dist[i] = np.inf
        penalty = np.maximum(0.0, self.repel_radius - dist)
        return float(np.sum(penalty))

    def step(self, z_agents):
        N, d = z_agents.shape
        assert d == self.d

        z_center = self._latent_center(z_agents)
        directions = np.zeros_like(z_agents)

        for i in range(N):
            zi = z_agents[i]
            to_center = unit_vector(z_center - zi)
            to_home = unit_vector(self.z_home[i] - zi)
            mem = unit_vector(self.memory[i]) if np.linalg.norm(self.memory[i]) > 0 else np.zeros(self.d)
            repel = self._repulsion_direction(z_agents, i)

            base = (
                self.w_c * to_center +
                self.w_h * to_home +
                self.w_m * mem +
                self.w_r * repel
            )
            if np.linalg.norm(base) > 0:
                base = unit_vector(base)
            else:
                base = np.zeros(self.d)

            best_score = -np.inf
            best_dir = np.zeros(self.d)
            best_z_cand = zi.copy()

            for _ in range(self.K):
                noise = self.rng.normal(size=self.d)
                noise = unit_vector(noise)
                canddir = unit_vector(0.7 * base + 0.3 * noise)
                z_cand = zi + self.step_size * canddir

                nov = self._novelty_score(z_cand, k=5)
                space = self._spacing_penalty(z_cand, z_agents, i)
                center_align = float(np.dot(canddir, to_center))

                score = (
                    self.lambda_nov * nov
                    - self.lambda_space * space
                    + self.lambda_center * center_align
                )

                if score > best_score:
                    best_score = score
                    best_dir = canddir
                    best_z_cand = z_cand

            new_dir = unit_vector(self.gamma * best_dir + (1.0 - self.gamma) * mem)
            directions[i] = new_dir
            self.memory[i] = new_dir
            self.archive.append(best_z_cand.copy())

        return directions

## 5 — Encoder and Four Rooms helpers

The encoder is the identity mapping (`z = state`).  We also define a helper to
**snap LHS-proposed positions to the nearest free cell** — necessary because LHS
may place agents inside walls.

In [ ]:
def encoder(states):
    """Identity encoder: z = state (continuous 2-D position)."""
    return np.array(states, dtype=np.float64).copy()


def snap_to_nearest_free_cell(pos, free_cells):
    """
    Given a continuous position `pos` (2,) and an array of free cells (M, 2),
    return the centre of the closest free cell.
    """
    centres = free_cells.astype(np.float32) + 0.5
    dists = np.linalg.norm(centres - pos[None, :], axis=1)
    idx = np.argmin(dists)
    return centres[idx]

## 6 — Episode collection helper

Each agent has its own `FourRoomsGridWorld` instance.  The planner provides
a 2-D direction vector; we scale it to an action in [-1, 1]² and call
`env.step()`.  The environment handles wall collisions internally via its
substep integration.

In [ ]:
def collect_steps_multi_agent(
    envs,
    planner,
    latent_mgr,
    num_steps=200,
    action_scale=1.0,
):
    """
    Run `num_steps` of coordinated multi-agent exploration.

    Returns
    -------
    all_transitions : list of (state, action, next_state) tuples
    all_positions   : list of np.ndarray snapshots (one per step)
    """
    num_agents = len(envs)

    # Current continuous positions (from env observations)
    positions = np.stack([env._agent_pos.copy() for env in envs])  # (N, 2)

    all_transitions = []
    all_positions = [positions.copy()]

    for t in range(num_steps):
        z_agents = encoder(positions)  # identity

        # Planner proposes unit-vector directions
        directions = planner.step(z_agents)  # (N, 2)

        new_positions = np.zeros_like(positions)
        for i in range(num_agents):
            state = positions[i].copy()
            # Scale direction to continuous action in [-1, 1]
            action = np.clip(action_scale * directions[i], -1.0, 1.0).astype(np.float32)
            obs, _, terminated, truncated, _ = envs[i].step(action)
            new_positions[i] = obs.copy()

            all_transitions.append((state, action, obs.copy()))

            # If the episode ended, reset the env
            if terminated or truncated:
                obs_reset, _ = envs[i].reset()
                new_positions[i] = obs_reset.copy()

        positions = new_positions
        all_positions.append(positions.copy())

    return all_transitions, all_positions

## 7 — Multi-agent exploration on Four Rooms

We use **20 agents × 200 steps = 4 000 total interactions**, matching the
dummy-grid experiment.  The planner hyper-parameters are the same except:

- `repel_radius = 2.0` (the grid is ≈ 2.3× larger)
- `step_size = 1.0` (one cell-width scale)

In [ ]:
def run_fourrooms_multi_agent_exploration(
    num_agents=20,
    latent_dim=2,
    num_steps=200,
    seed=0,
):
    rng = np.random.RandomState(seed)

    # Create one reference env for metadata
    ref_env = FourRoomsGridWorld()
    gs = ref_env.grid_size  # 23
    free_cells = ref_env._free_cells  # (M, 2)

    # ── LatentSpaceManager ──
    latent_mgr = LatentSpaceManager(latent_dim=latent_dim, init_c=2.0, rng=rng)

    # Bootstrap stats from free-cell centres
    bootstrap_states = free_cells.astype(np.float64) + 0.5
    latent_mgr.update_stats(encoder(bootstrap_states))

    # ── LHS initial positions, snapped to free cells ──
    z_init = latent_mgr.lhs_initial_latents(num_agents)  # (N, 2)
    start_positions = []
    for z in z_init:
        snapped = snap_to_nearest_free_cell(z, free_cells)
        start_positions.append(snapped)
    start_positions = np.stack(start_positions)  # (N, 2)

    # ── Create per-agent environments ──
    envs = []
    for i in range(num_agents):
        env = FourRoomsGridWorld()
        env.reset(seed=seed + i, options={'start_position': start_positions[i]})
        envs.append(env)

    # ── Planner ──
    planner = LatentExplorationPlanner(
        latent_dim=latent_dim,
        num_candidates=16,
        center_weight=0.1,
        home_weight=0.1,
        memory_weight=0.3,
        repel_weight=0.3,
        novelty_weight=1.0,
        spacing_weight=1.0,
        center_align_weight=0.1,
        memory_smooth=0.7,
        repel_radius=2.0,   # larger grid → larger radius
        step_size=1.0,      # one-cell scale
        rng=rng,
    )
    planner.reset_agents(encoder(start_positions))

    # ── Run exploration ──
    transitions, position_history = collect_steps_multi_agent(
        envs, planner, latent_mgr, num_steps=num_steps, action_scale=1.0
    )

    # ── Build coverage grid ──
    visited = np.zeros((gs, gs), dtype=np.int32)
    for pos_snapshot in position_history:
        for pos in pos_snapshot:
            cx, cy = int(np.floor(pos[0])), int(np.floor(pos[1]))
            cx = np.clip(cx, 0, gs - 1)
            cy = np.clip(cy, 0, gs - 1)
            visited[cx, cy] += 1

    print(f'Multi-agent exploration: {len(transitions)} transitions collected')
    return visited, transitions, position_history, ref_env


multi_visited, multi_transitions, multi_pos_history, ref_env = (
    run_fourrooms_multi_agent_exploration()
)

## 8 — Single-agent random-walk baseline

For a fair comparison we give the single agent the **same total budget**
(4 000 steps across 20 episodes of 200 steps each).

In [ ]:
def run_fourrooms_single_agent_random_walk(
    num_steps=200,
    num_episodes=20,
    seed=42,
):
    rng = np.random.RandomState(seed)
    env = FourRoomsGridWorld()
    gs = env.grid_size

    visited = np.zeros((gs, gs), dtype=np.int32)
    all_transitions = []

    for ep in range(num_episodes):
        obs, _ = env.reset(seed=seed + ep)

        for t in range(num_steps):
            # Random direction as continuous action
            noise = rng.normal(size=2).astype(np.float32)
            norm = np.linalg.norm(noise) + 1e-8
            action = np.clip(noise / norm, -1.0, 1.0)

            state = obs.copy()
            obs, _, terminated, truncated, _ = env.step(action)
            all_transitions.append((state, action, obs.copy()))

            cx = np.clip(int(np.floor(obs[0])), 0, gs - 1)
            cy = np.clip(int(np.floor(obs[1])), 0, gs - 1)
            visited[cx, cy] += 1

            if terminated or truncated:
                obs, _ = env.reset(seed=seed + ep * 1000 + t)

    print(f'Single-agent random walk: {len(all_transitions)} transitions collected')
    return visited, all_transitions


single_visited, single_transitions = run_fourrooms_single_agent_random_walk()

## 9 — Coverage comparison with wall overlay

We overlay the Four Rooms walls on top of the visit-count heatmaps so it is
easy to see which rooms and corridors are explored.

In [ ]:
def plot_coverage_comparison(multi_visited, single_visited, ref_env):
    gs = ref_env.grid_size

    # Build wall mask (1 where blocked)
    wall_mask = np.zeros((gs, gs), dtype=np.float32)
    for (wx, wy) in ref_env._blocked_cells:
        if 0 <= wx < gs and 0 <= wy < gs:
            wall_mask[wx, wy] = 1.0

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for ax, data, title in [
        (axes[0], multi_visited, 'Multi-agent (planner)'),
        (axes[1], single_visited, 'Single-agent (random walk)'),
    ]:
        im = ax.imshow(data.T, origin='lower', extent=[0, gs, 0, gs],
                       cmap='viridis', interpolation='nearest')
        # Overlay walls in red
        wall_rgba = np.zeros((gs, gs, 4))
        wall_rgba[:, :, 0] = wall_mask        # red channel
        wall_rgba[:, :, 3] = wall_mask * 0.7  # alpha
        ax.imshow(wall_rgba.transpose(1, 0, 2), origin='lower',
                  extent=[0, gs, 0, gs], interpolation='nearest')
        ax.set_title(title)
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        plt.colorbar(im, ax=ax, label='Visit count')

    # Difference map
    diff = multi_visited.astype(float) - single_visited.astype(float)
    vabs = max(abs(diff.min()), abs(diff.max()), 1)
    im = axes[2].imshow(diff.T, origin='lower', extent=[0, gs, 0, gs],
                        cmap='RdBu', vmin=-vabs, vmax=vabs,
                        interpolation='nearest')
    wall_rgba2 = np.zeros((gs, gs, 4))
    wall_rgba2[:, :, 0] = wall_mask
    wall_rgba2[:, :, 3] = wall_mask * 0.5
    axes[2].imshow(wall_rgba2.transpose(1, 0, 2), origin='lower',
                   extent=[0, gs, 0, gs], interpolation='nearest')
    axes[2].set_title('Difference (multi − single)')
    axes[2].set_xlabel('x')
    plt.colorbar(im, ax=axes[2], label='Δ visits')

    plt.suptitle('Four Rooms coverage: multi-agent planner vs single-agent random walk',
                 fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()


plot_coverage_comparison(multi_visited, single_visited, ref_env)

## 10 — Coverage metrics

In [ ]:
def coverage_fraction(visited, ref_env):
    """Fraction of *free* cells visited at least once."""
    gs = ref_env.grid_size
    free_mask = np.ones((gs, gs), dtype=bool)
    for (wx, wy) in ref_env._blocked_cells:
        if 0 <= wx < gs and 0 <= wy < gs:
            free_mask[wx, wy] = False
    visited_free = (visited > 0) & free_mask
    return visited_free.sum() / free_mask.sum()


multi_cov = coverage_fraction(multi_visited, ref_env)
single_cov = coverage_fraction(single_visited, ref_env)

print(f'Multi-agent free-cell coverage:  {multi_cov:.2%}')
print(f'Single-agent free-cell coverage: {single_cov:.2%}')
print(f'Advantage: {multi_cov - single_cov:+.2%}')

## 11 — Discussion and next steps

### What we showed

The multi-agent exploration planner that was validated on a simple open grid
transfers directly to the **Four Rooms GridWorld** — an environment with walls
and narrow doorways.  By coordinating 20 agents via the
`LatentExplorationPlanner` (novelty + repulsion + centering), we achieve
significantly higher free-cell coverage than a single agent performing a random
walk with the same total step budget.

### Key observations

1. **Walls are handled transparently** by the env's substep collision logic —
   the planner itself did not need any wall-awareness.
2. **LHS initialization + free-cell snapping** ensures agents start spread
   across all four rooms from the very first step.
3. The collected `(s, a, s')` transitions form a high-coverage replay buffer
   that is ideal for training the Deep Option Network.

### Next steps for DON training

- Swap the identity encoder with a learned forward-backward (FB)
  representation and re-run exploration in latent space.
- Feed the collected transitions into a `TrajectoryReplayBuffer` (see
  `src/utils.py`) for offline DON training.
- Periodically re-run exploration with the updated encoder to refine
  coverage.